# OP-01 · Téléchargement et contrôle qualité C3S

**Notebook opérationnel** — à exécuter chaque mois, au début du cycle.

| | |
|---|---|
| Étape du workflow | E2 — acquisition des sorties de modèles |
| Entrée | `config/cycle_YYYYMM.yaml` |
| Sorties | `DATA_OSF/raw/c3s/YYYYMM/*.grib` · `OUTPUTS_OSF/qc/YYYYMM/c3s_<variable>_*.csv` · manifestes dans `OUTPUTS_OSF/runs/` |
| Durée | 3 à 4 h au premier passage (file d'attente du CDS) ; quelques minutes ensuite |

**Mode d'emploi :** renseigner la cellule *Paramètres*, puis *Exécuter tout*. Les fichiers déjà téléchargés ne sont pas retéléchargés. Pour relire les requêtes sans rien télécharger, mettre `DRY_RUN = True`.

Équivalent en ligne de commande :
```
python scripts/run_download_c3s.py --config config/cycle_YYYYMM.yaml --variable precip
python scripts/run_qc_c3s.py       --config config/cycle_YYYYMM.yaml --variable precip
```

## Paramètres

In [ ]:
CYCLE_CONFIG = "config/cycle_202611.yaml"   # fichier du cycle du mois
VARIABLES    = ["precip"]                   # une liste, téléchargée variable après variable :
                                            # "precip", "tmoy" (= "t2m"), "tmax", "tmin"
MODELS       = None                         # None = tous les modèles de la configuration, ou ["ecmwf", "dwd"]
KINDS        = ["forecast", "hindcast"]
STREAM       = "auto"                       # "auto" suit precip_from ; "daily" ou "monthly" force un flux
DRY_RUN      = False                        # True : afficher les requêtes sans télécharger

In [ ]:
from pathlib import Path
import os, json
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)

from eccas_s2s.settings import load_cycle
from eccas_s2s.operations import download_c3s, qc_c3s
from eccas_s2s.core.periods import build_periods, periods_table

cfg = load_cycle(CYCLE_CONFIG)
print(f"Cycle {cfg.cycle_id} — initialisation {cfg.init_date.date()} — {len(cfg.c3s_models)} modèles C3S")
print("Données brutes :", cfg.raw_dir("c3s"))

## 1. Téléchargement

In [ ]:
dls = download_c3s.run_variables(CYCLE_CONFIG, VARIABLES, kinds=KINDS, models=MODELS,
                                 dry_run=DRY_RUN, stream=STREAM)
print()
for variable, dl in dls.items():
    if isinstance(dl, Exception):
        print(f"{variable:7s} ÉCHEC : {dl}")
        continue
    failures = dl.parameters.get("failures", {})
    print(f"{variable:7s} {dl.status} | {len(dl.outputs)} fichier(s) | {len(failures)} échec(s)")
    for what, err in failures.items():
        print("   échec :", what, "->", err)

### Deux flux de pluie, selon le modèle

Depuis le 23/09/2026, chaque modèle reçoit le ou les flux déclarés par sa clé
`precip_from` dans la configuration :

| modèle | décades | mois et saisons |
|---|---|---|
| ECMWF, Météo-France, DWD, CMCC, ECCC | journalier | journalier |
| UKMO, BoM | journalier | **mensuel** |
| NCEP CFSv2 | *(pas de hindcast journalier)* | **mensuel** |

Raison : l'archive journalière ne contient, pour les systèmes à démarrages
décalés, que les membres du 1er du mois — UKMO 7 membres en hindcast contre 28
dans l'archive mensuelle, BoM 3 contre 27, NCEP 0 contre 24. Le tableau ci-dessous
compte les membres **du flux téléchargé**.

## 2. Contrôle qualité des fichiers

Pour chaque fichier : nombre de membres et d'années, échéances reçues, horizon disponible pour **toutes** les années, échéances manquantes. UKMO et BoM ont peu de membres au 1er du mois (ensembles décalés). Ils sont conservés tels quels (décision du 19/09/2026) et signalés ici.

In [ ]:
if DRY_RUN:
    print("DRY_RUN = True : contrôle qualité non exécuté.")
else:
    # une table par variable et par flux : journalier (pluie, Tmax, Tmin) et mensuel
    # (pluie de UKMO, BoM et NCEP ; T2m de tous les modèles)
    qcs = qc_c3s.run_variables(CYCLE_CONFIG, VARIABLES)
    cols = {"daily": ["centre", "system", "kind", "members", "years", "n_steps",
                      "horizon_all_years_days", "horizon_any_year_days", "size_mb", "warnings"],
            "monthly": ["centre", "system", "kind", "members", "members_min", "start_dates",
                        "years", "n_steps", "horizon_all_years_months", "size_mb", "warnings"]}
    for variable, (qc, horizons, qc_ctx) in qcs.items():
        for stream, g in qc.groupby("stream"):
            print(f"— {variable} · flux {stream}")
            display(g[cols[stream]])

## 3. Horizon utilisable et cohérence avec la configuration

In [ ]:
if not DRY_RUN:
    lead_days = {c: m.max_lead_days for c, m in cfg.c3s_models.items()}
    for variable, (qc, horizons, qc_ctx) in qcs.items():
        h = horizons.reindex(columns=[*horizons.columns,
                                      *[c for c in ("usable_days", "usable_months")
                                        if c not in horizons.columns]])
        # journalier : l'horizon reçu doit égaler max_lead_days ;
        # mensuel : il doit couvrir l'horizon commun en mois
        h["attendu (config)"] = [lead_days[c] if s == "daily" else cfg.max_lead_months
                                 for c, s in zip(h["centre"], h["stream"])]
        h["cohérent"] = [bool(d == a) if s == "daily" else bool(m >= a)
                         for s, d, m, a in zip(h["stream"], h["usable_days"],
                                               h["usable_months"], h["attendu (config)"])]
        print(f"— {variable}")
        display(h)
        if not h["cohérent"].all():
            print("⚠ Horizon incohérent : mettre à jour max_lead_days dans", CYCLE_CONFIG,
                  "ou vérifier le téléchargement mensuel.")
        if qc_ctx.warnings:
            print("\nAlertes du contrôle qualité :")
            for w in qc_ctx.warnings:
                print(" -", w)

## 4. Périodes produites avec l'horizon commun

`cfg.periods_for(...)` applique le plafond de la configuration (`horizon.max_lead_months`,
6 mois d'échéance : le CDS ne propose pas le 7ᵉ mois pour UKMO, BoM et NCEP), puis le
sélecteur. Ici `selection="all"` montre tout ce que le cycle peut produire ; les étapes
suivantes prennent `--periods all-seasons`, `--periods "OND 2026"`, etc., ou rien du tout
pour rester sur le sous-ensemble de développement.


In [ ]:
if not DRY_RUN and qcs:
    # horizon commun du flux journalier (premier jeu de variables qui en a un)
    jours = [h.loc[h["stream"] == "daily", "usable_days"] for (_, h, _) in qcs.values()
             if "usable_days" in h.columns]
    jours = pd.concat(jours).dropna() if jours else pd.Series(dtype="float")
    common = int(jours.min()) if len(jours) else None
    if common:
        print(f"Horizon commun (journalier) : {common} jours → "
              f"jusqu'au {(cfg.init_date + pd.Timedelta(days=common - 1)).date()}")
    else:
        print("Aucun flux journalier contrôlé : seules les périodes du flux mensuel comptent.")
    print(f"Plafond de la configuration : {cfg.max_lead_months} mois d'échéance")
    tab = periods_table(cfg.periods_for(common or 10_000, cfg.scales, selection="all"),
                        cfg.init_date.year)
    display(tab.groupby("scale")["label"].agg(["count", "first", "last"]))

## 5. Traçabilité

In [ ]:
contexts = [c for c in dls.values() if not isinstance(c, Exception)]
if not DRY_RUN:
    contexts += [ctx for (_, _, ctx) in qcs.values()]
for c in contexts:
    print(f"{c.step:24s} {c.status:8s} {c.run_dir / 'manifest.json'}")